<a href="https://colab.research.google.com/github/onomtonks/RAG-APP/blob/master/src/chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 6.3 MB/s eta 0:00:00


In [ ]:
%pip install pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 84.3 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF

doc = fitz.open("extracted_page_3.pdf")

# structure tree is stored under the PDF catalog
catalog_entry = doc.pdf_catalog()

# Check if the catalog_entry is a dictionary before trying to access keys
if isinstance(catalog_entry, dict) and "StructTreeRoot" in catalog_entry:
    tree = catalog_entry["StructTreeRoot"]
    print(tree)
else:
    print("PDF has no structure tags or the PDF catalog entry is not a dictionary.")

PDF has no structure tags or the PDF catalog entry is not a dictionary.


In [ ]:
pip install pikepdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 24.0 MB/s eta 0:00:00


In [ ]:
import pikepdf

try:
    pdf = pikepdf.open("extracted_page_3.pdf")

    # In pikepdf, pdf.root is the standard way to get the root dictionary.
    # The AttributeError suggests it might be missing or corrupted.
    # We'll use a try-except block to gracefully handle this.
    try:
        root = pdf.root
        if "/StructTreeRoot" in root:
            struct = root["/StructTreeRoot"]
            print(struct)
        else:
            print("No StructTreeRoot tag in PDF's root dictionary.")
    except AttributeError:
        print("Error: The pikepdf.Pdf object does not have a 'root' attribute. This often indicates a malformed PDF or a problem during parsing.")
    finally:
        pdf.close() # Ensure the PDF is closed
except pikepdf.errors.PdfError as e:
    print(f"Error opening or processing PDF with pikepdf: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Error: The pikepdf.Pdf object does not have a 'root' attribute. This often indicates a malformed PDF or a problem during parsing.


In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        print(full_text)
        full_text += page.extract_text() + "\n"  # Add a newline between pages
    return full_text

pdf_file_path = "extracted_page_3.pdf"  # Replace with your PDF file path
extracted_text = extract_text_from_pdf(pdf_file_path)
print(extracted_text)


1.1 Pre-training NLP Models 3
representations. In pre-training, this model is combined with a classification layer to form a clas-
sification system. This system is then trained on a pre-training task, such as classifying sentences
based on sentiment (e.g., determining if a sentence conveys a positive or negative sentiment).
Then, we adapt the sequence model to a downstream task. We build a new classification system
based on this pre-trained sequence model and a new classification layer (e.g., determining if a
sequence is subjective or objective). Typically, we need to fine-tune the parameters of the new
model using task-specific labeled data, ensuring the model is optimally adjusted to perform well
on this new type of data. The fine-tuned model is then employed to classify new sequences for
this task. An advantage of supervised pre-training is that the training process, either in the pre-
training or fine-tuning phase, is straightforward, as it follows the well-studied general paradi

In [ ]:
import re

def separate_with_regex(text):
    # Regex pattern: Matches one or more whitespace characters (\s)
    # followed by two or more newline characters, potentially with more
    # whitespace between them. This captures true paragraph breaks.
    # 're.split' returns a list of strings split by the pattern.
    paragraphs = re.split(r'\s*\n{2,}\s*', text)

    # Clean up the list by stripping and removing empty elements
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    return paragraphs

# Assuming 'extracted_text' is the text you got from your function:
# extracted_text = "..."
regex_paragraph_list = separate_with_regex(extracted_text)

In [ ]:
regex_paragraph_list

IndexError: list index out of range

In [ ]:
def separate_by_indentation(text):
    # Replace all occurrences of one newline followed by a space (common
    # pattern for a new line that is NOT a new paragraph) with just a space.
    # This might merge accidentally split lines.
    cleaned_text = re.sub(r'(?<!\n)\n(?![\n\s])', ' ', text)

    # Now, split by two or more newlines (the standard paragraph break)
    paragraphs = separate_by_multiple_newlines(cleaned_text)
    return paragraphs

In [ ]:
def extract_paragraphs(text):
    # Split by two or more newlines to identify potential paragraph breaks
    paragraphs = [p.strip() for p in text.split('.\n') if p.strip()]
    return paragraphs

paragraphs = extract_paragraphs(extracted_text)
for i, paragraph in enumerate(paragraphs):
    print(f"Paragraph {i+1}:\n{paragraph}\n---")

Paragraph 1:
1.1 Pre-training NLP Models 3
representations. In pre-training, this model is combined with a classification layer to form a clas-
sification system. This system is then trained on a pre-training task, such as classifying sentences
based on sentiment (e.g., determining if a sentence conveys a positive or negative sentiment)
---
Paragraph 2:
Then, we adapt the sequence model to a downstream task. We build a new classification system
based on this pre-trained sequence model and a new classification layer (e.g., determining if a
sequence is subjective or objective). Typically, we need to fine-tune the parameters of the new
model using task-specific labeled data, ensuring the model is optimally adjusted to perform well
on this new type of data. The fine-tuned model is then employed to classify new sequences for
this task. An advantage of supervised pre-training is that the training process, either in the pre-
training or fine-tuning phase, is straightforward, as it follows the

In [ ]:
print(paragraphs[0])

1.1 Pre-training NLP Models 3
representations. In pre-training, this model is combined with a classification layer to form a clas-
sification system. This system is then trained on a pre-training task, such as classifying sentences
based on sentiment (e.g., determining if a sentence conveys a positive or negative sentiment)


In [ ]:
from pypdf import PdfReader, PdfWriter

def extract_single_page(input_pdf_path, page_number_to_extract, output_pdf_path):
    """
    Extracts a single page from a PDF, saves it as a new PDF,
    and returns the text of that page.
    """
    try:
        reader = PdfReader(input_pdf_path)
        writer = PdfWriter()

        # Ensure page number is valid
        if 1 <= page_number_to_extract <= len(reader.pages):
            page = reader.pages[page_number_to_extract - 1]

            # Save page to new PDF
            writer.add_page(page)
            with open(output_pdf_path, "wb") as f:
                writer.write(f)

            # Return extracted text
            return page.extract_text()
        else:
            print("Invalid page number.")
            return None

    except FileNotFoundError:
        print(f"Error: Input PDF file not found at {input_pdf_path}")
    except Exception as e:
        print(f"An error occurred: {e}")

    return None


In [ ]:
text = extract_single_page("extracted_page_3.pdf", 10, "extracted_page_10.pdf")

if text:
    print(text)
else:
    print("Text extraction failed.")


Invalid page number.
Text extraction failed.


In [ ]:
%pip install sentence-transformers pypdf hdbscan scikit-learn numpy

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import hdbscan
from pypdf import PdfReader


def extract_paragraphs_from_pdf(path):
    reader = PdfReader(path)
    text = ""

    for page in reader.pages:
        text += page.extract_text() + "\n"

    # Split paragraphs by single newlines to get more segments
    paragraphs = [p.strip() for p in text.split('.\n') if p.strip()]
    return paragraphs


pdf_path = "llm-book.pdf"
paragraphs = extract_paragraphs_from_pdf(pdf_path)
print("Paragraphs:", len(paragraphs))


model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(
    paragraphs,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)


pca = PCA(n_components=50)
reduced_embeddings = pca.fit_transform(embeddings)

print("Reduced shape:", reduced_embeddings.shape)


clusterer = hdbscan.HDBSCAN(
    min_cluster_size=3,
    metric='euclidean'
)

labels = clusterer.fit_predict(reduced_embeddings)

print("Clusters found:", set(labels))


clusters = {}
for idx, label in enumerate(labels):
    if label == -1:
        # -1 = noise; you can skip or handle separately
        continue
    clusters.setdefault(label, []).append((idx, paragraphs[idx]))

# Sort paragraphs by original PDF order
for label in clusters:
    clusters[label] = [p for _, p in sorted(clusters[label], key=lambda x: x[0])]


chunks = []
for label, paras in clusters.items():
    chunk = ".\n".join(paras)
    chunks.append((label, chunk))

print(f"Generated {len(chunks)} chunks")

chunk_texts = [c[1] for c in chunks]
chunk_embeddings = model.encode(chunk_texts, normalize_embeddings=True)

print("Final chunk embeddings:", chunk_embeddings.shape)

for label, chunk in chunks[:3]:
    print(f"\n=== Chunk {label} ===\n{chunk[:500]}...\n")

Paragraphs: 1896
Embedding shape: (1896, 768)
Reduced shape: (1896, 50)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Clusters found: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64(70),

In [ ]:
label, text = chunks[50]
print(text)

For LLM prompting, it is also possible to improve performance by combining predictions
based on different prompts. Suppose we have an LLM and a collection of prompts that address
the same task. We can run this LLM with each of the prompts and then combine the predictions.
Formally, let{x1,..., xK}be Kprompts for performing the same task. Given an LLMPr(·|·),
we can find the best prediction for each xi using ˆyi = arg maxyi Pr(yi|xi). These predictions
can be combined to form a “new” prediction:
ˆy = Combine( ˆy1,..., ˆyK) (3.6)
Here Combine(·) is the combination model, which can be designed in several different ways. For
example, we can select the best prediction by voting or by identifying the one that overlaps the
most with others. Another method for model combination is to perform model averaging during
token prediction. Let ˆyj be the predicted token at the j-th step for model combination. The
probability of predicting ˆyj is given by
ˆyj = arg max
yj
K∑
k=1
log Pr(yj|xk,ˆy1,..., ˆ

In [ ]:
import json

# Convert chunks into a serializable structure
chunks_json = [
    {"cluster_id": int(label), "text": chunk}
    for label, chunk in chunks
]

# Save to file
output_path = "final_chunks.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks_json, f, indent=4, ensure_ascii=False)

print(f"Saved {len(chunks_json)} chunks to {output_path}")

Saved 72 chunks to final_chunks.json
